# 🧮 LeetCode 高频题精选 —— LLM 系统关联版> 每道题都标注了与 LLM 系统的关联，面试时可以自然延伸。> **建议**：每题限时 20 分钟，先写思路再写代码。

## 1. LRU Cache (LC 146) ⭐⭐⭐**LLM 关联**：KV Cache 驱逐策略的核心数据结构- 时间复杂度：get/put O(1)- 关键：双向链表 + 哈希表

In [ ]:
class Node:
    __slots__ = ('key', 'val', 'prev', 'next')
    def __init__(self, key=0, val=0):
        self.key, self.val = key, val
        self.prev = self.next = None

class LRUCache:
    """手写双向链表版 LRU，不用 OrderedDict（面试官更喜欢）"""
    def __init__(self, capacity: int):
        self.cap = capacity
        self.map = {}
        self.head, self.tail = Node(), Node()  # dummy
        self.head.next = self.tail
        self.tail.prev = self.head

    def _remove(self, node: Node):
        node.prev.next = node.next
        node.next.prev = node.prev

    def _add_to_front(self, node: Node):
        node.next = self.head.next
        node.prev = self.head
        self.head.next.prev = node
        self.head.next = node

    def get(self, key: int) -> int:
        if key not in self.map:
            return -1
        node = self.map[key]
        self._remove(node)
        self._add_to_front(node)
        return node.val

    def put(self, key: int, value: int) -> None:
        if key in self.map:
            self._remove(self.map[key])
        node = Node(key, value)
        self.map[key] = node
        self._add_to_front(node)
        if len(self.map) > self.cap:
            lru = self.tail.prev
            self._remove(lru)
            del self.map[lru.key]

# ---- 测试 ----
cache = LRUCache(2)
cache.put(1, 1); cache.put(2, 2)
assert cache.get(1) == 1
cache.put(3, 3)  # evicts key 2
assert cache.get(2) == -1
cache.put(4, 4)  # evicts key 1
assert cache.get(1) == -1
assert cache.get(3) == 3
assert cache.get(4) == 4
print("✅ LRU Cache: all tests passed")


### 面试延伸：LRU → Paged KV Cache 驱逐```面试回答模板："LRU 在 KV Cache 中不是按 key-value 对驱逐，而是按 block 粒度。 一个 prefix 可能占用多个 block，驱逐时需要释放整个 prefix 的所有 block。 此外，实际系统中还需要考虑 reference counting（prefix sharing）和 多租户公平性（quota-aware eviction）。"```

## 2. LFU Cache (LC 460) ⭐⭐⭐**LLM 关联**：热门 prefix 缓存策略，频率驱逐 vs 时间驱逐的权衡

In [ ]:
from collections import defaultdict

class LFUCache:
    def __init__(self, capacity: int):
        self.cap = capacity
        self.min_freq = 0
        self.key_val = {}
        self.key_freq = {}
        self.freq_keys = defaultdict(dict)  # freq -> {key: None} (ordered by insertion)
        self.time = 0  # tie-break by time

    def _update(self, key):
        freq = self.key_freq[key]
        del self.freq_keys[freq][key]
        if not self.freq_keys[freq]:
            del self.freq_keys[freq]
            if self.min_freq == freq:
                self.min_freq += 1
        self.key_freq[key] = freq + 1
        self.freq_keys[freq + 1][key] = self.time
        self.time += 1

    def get(self, key: int) -> int:
        if key not in self.key_val:
            return -1
        self._update(key)
        return self.key_val[key]

    def put(self, key: int, value: int) -> None:
        if self.cap <= 0:
            return
        if key in self.key_val:
            self.key_val[key] = value
            self._update(key)
            return
        if len(self.key_val) >= self.cap:
            # evict LFU (LRU among ties)
            evict_key = next(iter(self.freq_keys[self.min_freq]))
            del self.freq_keys[self.min_freq][evict_key]
            if not self.freq_keys[self.min_freq]:
                del self.freq_keys[self.min_freq]
            del self.key_val[evict_key]
            del self.key_freq[evict_key]
        self.key_val[key] = value
        self.key_freq[key] = 1
        self.freq_keys[1][key] = self.time
        self.time += 1
        self.min_freq = 1

# ---- 测试 ----
lfu = LFUCache(2)
lfu.put(1, 1); lfu.put(2, 2)
assert lfu.get(1) == 1     # freq(1)=2, freq(2)=1
lfu.put(3, 3)              # evicts key 2 (lowest freq)
assert lfu.get(2) == -1
assert lfu.get(3) == 3
lfu.put(4, 4)              # evicts key 1 or 3, key 3 freq=2, key 1 freq=2 -> LRU: key 1
assert lfu.get(1) == -1
print("✅ LFU Cache: all tests passed")


## 3. Top-K Frequent Elements (LC 347) ⭐⭐**LLM 关联**：Top-K Sampling / Token 频率统计 / 热门 Prefix 分析

In [ ]:
import heapq
from collections import Counter

def top_k_frequent(nums, k):
    """O(n log k) 用最小堆"""
    count = Counter(nums)
    return heapq.nlargest(k, count.keys(), key=count.get)

def top_k_frequent_bucket(nums, k):
    """O(n) 桶排序法 —— 面试加分"""
    count = Counter(nums)
    buckets = [[] for _ in range(len(nums) + 1)]
    for num, freq in count.items():
        buckets[freq].append(num)
    result = []
    for i in range(len(buckets) - 1, -1, -1):
        for num in buckets[i]:
            result.append(num)
            if len(result) == k:
                return result
    return result

# ---- 测试 ----
assert set(top_k_frequent([1,1,1,2,2,3], 2)) == {1, 2}
assert set(top_k_frequent_bucket([1,1,1,2,2,3], 2)) == {1, 2}
print("✅ Top-K Frequent: all tests passed")


## 4. 滑动窗口最大值 (LC 239) ⭐⭐⭐**LLM 关联**：Sliding Window Attention / KV Cache 窗口驱逐策略

In [ ]:
from collections import deque

def max_sliding_window(nums, k):
    """单调递减队列 O(n)"""
    dq = deque()  # 存 index，保持值递减
    result = []
    for i, v in enumerate(nums):
        # 移除窗口外的
        while dq and dq[0] < i - k + 1:
            dq.popleft()
        # 保持递减
        while dq and nums[dq[-1]] <= v:
            dq.pop()
        dq.append(i)
        if i >= k - 1:
            result.append(nums[dq[0]])
    return result

# ---- 测试 ----
assert max_sliding_window([1,3,-1,-3,5,3,6,7], 3) == [3,3,5,5,6,7]
assert max_sliding_window([1], 1) == [1]
assert max_sliding_window([1,-1], 1) == [1,-1]
print("✅ Sliding Window Maximum: all tests passed")


### 面试延伸：Sliding Window → Sliding Window Attention```"滑动窗口最大值用的单调队列思想，和 Sliding Window Attention 的思路一致： 只关注最近的 W 个 token，超出窗口的 KV 直接驱逐。 Mistral/Mixtral 使用 sliding window size = 4096，每层看不同的窗口， 堆叠多层后有效感受野 = layers × window_size。"```

## 5. Implement Trie (LC 208) ⭐⭐⭐**LLM 关联**：Radix Tree 用于 Prefix Caching / RAG 前缀复用

In [ ]:
class TrieNode:
    __slots__ = ('children', 'is_end')
    def __init__(self):
        self.children = {}
        self.is_end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word: str) -> None:
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.is_end = True

    def search(self, word: str) -> bool:
        node = self._find(word)
        return node is not None and node.is_end

    def starts_with(self, prefix: str) -> bool:
        return self._find(prefix) is not None

    def _find(self, prefix: str):
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return None
            node = node.children[ch]
        return node

# ---- 测试 ----
trie = Trie()
trie.insert("apple")
assert trie.search("apple") == True
assert trie.search("app") == False
assert trie.starts_with("app") == True
trie.insert("app")
assert trie.search("app") == True
print("✅ Trie: all tests passed")


### Trie → Radix Tree 延伸```python# Radix Tree 是 Trie 的压缩版：将只有一个孩子的连续节点合并# SGLang 的 RadixAttention 用 Radix Tree 管理 Prefix KV Cache：#   - 每个节点存储一段 token prefix 的 KV block IDs#   - 公共前缀只存一份，多个请求共享#   - 插入/查找/驱逐都是 O(prefix_length) #   - 比 hash table 更节省内存（公共前缀合并）```

## 6. 合并 K 个有序链表 (LC 23) ⭐⭐**LLM 关联**：多路归并 → 多 Worker 输出归并 / Top-K Sampling 合并

In [ ]:
import heapq

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

def merge_k_lists(lists):
    """最小堆归并 O(N log K)"""
    heap = []
    for i, node in enumerate(lists):
        if node:
            heapq.heappush(heap, (node.val, i, node))
    dummy = tail = ListNode()
    while heap:
        val, i, node = heapq.heappop(heap)
        tail.next = node
        tail = tail.next
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next

# ---- 测试 ----
def to_list(lst):
    dummy = cur = ListNode()
    for v in lst:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

def from_list(node):
    result = []
    while node:
        result.append(node.val)
        node = node.next
    return result

lists = [to_list([1,4,5]), to_list([1,3,4]), to_list([2,6])]
assert from_list(merge_k_lists(lists)) == [1,1,2,3,4,4,5,6]
print("✅ Merge K Sorted Lists: all tests passed")


## 7. 数据流中位数 (LC 295) ⭐⭐**LLM 关联**：实时 P50/P99 延迟监控

In [ ]:
import heapq

class MedianFinder:
    """两个堆：max_heap (左半) + min_heap (右半)"""
    def __init__(self):
        self.lo = []  # max-heap (negated)
        self.hi = []  # min-heap

    def addNum(self, num: int) -> None:
        heapq.heappush(self.lo, -num)
        heapq.heappush(self.hi, -heapq.heappop(self.lo))
        if len(self.hi) > len(self.lo):
            heapq.heappush(self.lo, -heapq.heappop(self.hi))

    def findMedian(self) -> float:
        if len(self.lo) > len(self.hi):
            return -self.lo[0]
        return (-self.lo[0] + self.hi[0]) / 2

# ---- 测试 ----
mf = MedianFinder()
mf.addNum(1); mf.addNum(2)
assert mf.findMedian() == 1.5
mf.addNum(3)
assert mf.findMedian() == 2.0
print("✅ Median Finder: all tests passed")


## 8. 设计哈希表 (LC 706) ⭐⭐**LLM 关联**：Token-to-ID mapping / Block Table 实现

In [ ]:
class MyHashMap:
    """链地址法，面试写 open addressing 也可以"""
    def __init__(self, size=1024):
        self.size = size
        self.buckets = [[] for _ in range(size)]

    def _hash(self, key):
        return key % self.size

    def put(self, key: int, value: int) -> None:
        bucket = self.buckets[self._hash(key)]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket[i] = (key, value)
                return
        bucket.append((key, value))

    def get(self, key: int) -> int:
        for k, v in self.buckets[self._hash(key)]:
            if k == key:
                return v
        return -1

    def remove(self, key: int) -> None:
        bucket = self.buckets[self._hash(key)]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket.pop(i)
                return

# ---- 测试 ----
hm = MyHashMap()
hm.put(1, 1); hm.put(2, 2)
assert hm.get(1) == 1
assert hm.get(3) == -1
hm.put(2, 1)
assert hm.get(2) == 1
hm.remove(2)
assert hm.get(2) == -1
print("✅ HashMap: all tests passed")


## 9. 一致性哈希 (系统设计编程题) ⭐⭐**LLM 关联**：分布式 Prefix Cache 路由 / SGLang consistent hashing

In [ ]:
import hashlib, bisect

class ConsistentHash:
    def __init__(self, nodes=None, replicas=100):
        self.replicas = replicas
        self.ring = []       # sorted hash values
        self.hash_to_node = {}
        for node in (nodes or []):
            self.add_node(node)

    def _hash(self, key):
        return int(hashlib.md5(key.encode()).hexdigest(), 16)

    def add_node(self, node):
        for i in range(self.replicas):
            h = self._hash(f"{node}:{i}")
            self.ring.append(h)
            self.hash_to_node[h] = node
        self.ring.sort()

    def remove_node(self, node):
        for i in range(self.replicas):
            h = self._hash(f"{node}:{i}")
            self.ring.remove(h)
            del self.hash_to_node[h]

    def get_node(self, key):
        if not self.ring:
            return None
        h = self._hash(key)
        idx = bisect.bisect_right(self.ring, h)
        if idx == len(self.ring):
            idx = 0
        return self.hash_to_node[self.ring[idx]]

# ---- 测试 ----
ch = ConsistentHash(["gpu-0", "gpu-1", "gpu-2"])
# 验证负载大致均衡
from collections import Counter
distribution = Counter()
for i in range(1000):
    distribution[ch.get_node(f"prefix-{i}")] += 1
print(f"Node distribution: {dict(distribution)}")
assert len(distribution) == 3
assert all(v > 200 for v in distribution.values())  # 大致均衡
print("✅ Consistent Hash: all tests passed")


## 10. 综合挑战：实现 Mini Token Scheduler ⭐⭐⭐**LLM 关联**：Continuous Batching + Preemption + Priority

In [ ]:
import heapq
from dataclasses import dataclass, field
from typing import List

@dataclass(order=True)
class Request:
    priority: int
    id: int = field(compare=False)
    prompt_len: int = field(compare=False)
    max_gen: int = field(compare=False)
    generated: int = field(default=0, compare=False)

    @property
    def done(self):
        return self.generated >= self.max_gen

class MiniScheduler:
    """Continuous batching + priority preemption."""
    def __init__(self, max_batch_tokens=512, max_batch_size=8):
        self.max_batch_tokens = max_batch_tokens
        self.max_batch_size = max_batch_size
        self.waiting: List[Request] = []   # min-heap by priority
        self.running: List[Request] = []
        self.completed: List[Request] = []

    def add_request(self, req: Request):
        heapq.heappush(self.waiting, req)

    def step(self):
        # 1. decode existing
        still_running = []
        for req in self.running:
            req.generated += 1
            if req.done:
                self.completed.append(req)
            else:
                still_running.append(req)
        self.running = still_running

        # 2. fill with waiting requests (prefill)
        decode_tokens = len(self.running)  # 1 token per running req
        budget = self.max_batch_tokens - decode_tokens
        while (self.waiting
               and len(self.running) < self.max_batch_size
               and budget > 0):
            req = heapq.heappop(self.waiting)
            if req.prompt_len <= budget:
                self.running.append(req)
                budget -= req.prompt_len
            else:
                heapq.heappush(self.waiting, req)
                break  # can't fit

        return len(self.completed)

    def run_to_completion(self):
        steps = 0
        while self.running or self.waiting:
            self.step()
            steps += 1
            if steps > 10000:
                raise RuntimeError("infinite loop")
        return steps

# ---- 测试 ----
sched = MiniScheduler(max_batch_tokens=256, max_batch_size=4)
sched.add_request(Request(priority=1, id=0, prompt_len=50, max_gen=10))
sched.add_request(Request(priority=0, id=1, prompt_len=30, max_gen=5))   # higher priority
sched.add_request(Request(priority=2, id=2, prompt_len=100, max_gen=20))

steps = sched.run_to_completion()
print(f"All requests completed in {steps} steps")
ids = [r.id for r in sched.completed]
print(f"Completion order: {ids}")
assert 1 in ids  # high priority should complete
assert len(sched.completed) == 3
print("✅ Mini Scheduler: all tests passed")


---## 📊 刷题进度跟踪| # | 题目 | 难度 | LLM关联 | 状态 ||---|------|------|---------|------|| 1 | LRU Cache | Medium | KV Cache 驱逐 | ⬜ || 2 | LFU Cache | Hard | 频率驱逐策略 | ⬜ || 3 | Top-K Frequent | Medium | Sampling/热prefix | ⬜ || 4 | Sliding Window Max | Hard | SWA/窗口驱逐 | ⬜ || 5 | Trie | Medium | Radix Prefix Cache | ⬜ || 6 | Merge K Lists | Hard | 多路归并 | ⬜ || 7 | Find Median | Hard | P50/P99 监控 | ⬜ || 8 | Design HashMap | Easy | Block Table | ⬜ || 9 | Consistent Hash | - | 分布式Cache路由 | ⬜ || 10 | Mini Scheduler | - | Continuous Batching | ⬜ |